In [3]:
"""
Builds the {query_id: gold_answer} mapping that finqa_answer_eval.py needs,
by joining FinanceRAG's queries.jsonl (BEIR format: _id, title, text) to the
original czyssrs/FinQA dataset's qa.question / qa.exe_ans, matched by
NORMALIZED QUESTION TEXT (the two datasets' IDs do not correspond).

Prerequisite: download train.json, dev.json, test.json from
https://github.com/czyssrs/FinQA/tree/main/dataset and note their local
paths below. (test.json has real references; the separate private_test.json
does not and should NOT be used as a gold source.)

Confirmed schema (from the FinQA README) -- each entry:
    {
      "id": ..., "pre_text": [...], "post_text": [...], "table": [...],
      "qa": {"question": ..., "program": ..., "gold_inds": ...,
             "exe_ans": ..., "program_re": ...}
    }

IMPORTANT: run this and read the printed match-rate summary before trusting
the output -- I have not verified this join against real file contents, so
the match rate is your evidence, not mine.
"""

import json
import re

FINQA_SOURCE_FILES = [
    "../finrag_data/finqa_source/train.json",
    "../finrag_data/finqa_source/dev.json",
    "../finrag_data/finqa_source/test.json",   # public test.json only -- NOT private_test.json
]

FINANCERAG_QUERIES_PATH = "../finrag_data/finqa_queries.jsonl/queries.jsonl"
OUTPUT_PATH = "../finrag_data/finqa_gold_answers.json"




In [4]:

def normalize(text: str) -> str:
    """Lowercase, collapse whitespace, strip trailing punctuation -- enough
    to absorb minor formatting differences between the two datasets without
    being so aggressive it creates false matches."""
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    text = text.rstrip("?.! ")
    return text




In [5]:
def load_finqa_gold(paths: list[str]) -> dict[str, float]:
    """Returns {normalized_question_text: exe_ans}."""
    gold_by_question = {}
    for path in paths:
        with open(path) as f:
            entries = json.load(f)
        for entry in entries:
            qa = entry.get("qa")
            if not qa or "question" not in qa or "exe_ans" not in qa:
                continue  # private_test-style entries with no reference
            try:
                exe_ans = float(qa["exe_ans"])
            except (TypeError, ValueError):
                continue  # non-numeric or malformed answer -- skip
            gold_by_question[normalize(qa["question"])] = exe_ans
    return gold_by_question


def build_gold_map(queries_path: str, gold_by_question: dict[str, float]) -> dict[str, float]:
    gold_map = {}
    unmatched = []
    with open(queries_path) as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            qid = str(row["_id"])
            norm_text = normalize(row["text"])
            if norm_text in gold_by_question:
                gold_map[qid] = gold_by_question[norm_text]
            else:
                unmatched.append(row["text"])

    total = len(gold_map) + len(unmatched)
    print(f"Matched {len(gold_map)} / {total} queries ({len(gold_map)/total:.1%})")
    if unmatched:
        print("\nSample unmatched queries (first 5) -- inspect these for near-misses")
        print("(extra whitespace, different question phrasing, currency formatting, etc.):")
        for q in unmatched[:5]:
            print(f"  - {q}")
    return gold_map

# ---------------------------------------------------------------------------
# ADDENDUM: exact-match audit. If build_gold_map's normalize() is too loose,
# it can match two DIFFERENT questions that happen to normalize identically
# (or near-identically) but aren't the same question -- run this to see how
# many of your matches are exact post-normalization vs. how many involved
# any actual difference worth eyeballing.
# ---------------------------------------------------------------------------
def audit_matches(queries_path: str, gold_by_question: dict[str, float], n_samples: int = 15):
    """Prints a sample of (FinanceRAG question, matched FinQA gold) pairs
    side by side so you can manually judge whether the join is matching the
    RIGHT question, not just A question."""
    import random
    with open(queries_path) as f:
        rows = [json.loads(line) for line in f if line.strip()]
 
    matched = [(row["text"], gold_by_question[normalize(row["text"])])
               for row in rows if normalize(row["text"]) in gold_by_question]
 
    print(f"\n=== Manual audit sample ({min(n_samples, len(matched))} of {len(matched)} matched pairs) ===")
    random.seed(0)
    for question, gold in random.sample(matched, min(n_samples, len(matched))):
        print(f"Q: {question}")
        print(f"gold: {gold}\n")
     

In [6]:
gold_by_question = load_finqa_gold(FINQA_SOURCE_FILES)
print(f"Loaded {len(gold_by_question)} gold Q&A pairs from FinQA source files")

gold_map = build_gold_map(FINANCERAG_QUERIES_PATH, gold_by_question)

with open(OUTPUT_PATH, "w") as f:
    json.dump([{"id": qid, "exe_ans": ans} for qid, ans in gold_map.items()], f)
print(f"\nWrote {len(gold_map)} matched pairs to {OUTPUT_PATH}")


Loaded 7965 gold Q&A pairs from FinQA source files
Matched 1127 / 1147 queries (98.3%)

Sample unmatched queries (first 5) -- inspect these for near-misses
(extra whitespace, different question phrasing, currency formatting, etc.):
  - was 2014 rent expense greater than 2015 operating lease expense?
  - was the five year total return of the 2019 peer group greater than the 2018 peer group?
  - did compensation expense related to the company 2019s employee stock purchase plan grow from 2004 to 2005?
  - did the firm cancel more stock options during 2017 than it repurchased in common shares?
  - did the company have more exposure to the insurance industry than the real estate industry in its derivative portfolio?

Wrote 1127 matched pairs to ../finrag_data/finqa_gold_answers.json


In [7]:
audit_matches(FINANCERAG_QUERIES_PATH, gold_by_question)


=== Manual audit sample (15 of 1127 matched pairs) ===
Q: considering the average exercise price of options , what is the increase in the total value of stock options observed during 2016 and 2017 , in millions of dollars?
gold: 16.43

Q: what was the percent of the jpmorgan chase income before income tax expense/ ( benefit ) and extraordinary gain that was us sourced
gold: 0.66648

Q: what percentage of total expected cash outflow to satisfy contractual obligations and commitments as of december 31 , 2010 are due in 2012?
gold: 0.05663

Q: for tax credit investments included in our equity investments held by consolidated partnerships , what was the change in billions between december 31 , 2014 and december 31 , 2013?
gold: 0.0

Q: what was map's 3 year growth of gasoline production?
gold: 0.03743

Q: for the vivo acquisition how many of the allowed towers were actually purchased under the final amended purchase agreement?
gold: 0.94

Q: what was the change in level 3 financial assets